<a href="https://colab.research.google.com/github/Maneshna/AI_LAB/blob/main/AILAB8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assignment 8: Navigation with Multiple Goals

Objective: Solve a problem where multiple goals exist using search algorithms.
Problem Statement: A robot in a grid needs to collect items (goals) before reaching an
exit. Each goal has a different priority or cost.
Tasks:
● Use BFS/DFS for simpler scenarios (unweighted goals).
● Implement A* or Uniform Cost Search for weighted scenarios.
● Analyze the trade-offs between path length and goal priority.

In [ ]:
import heapq
from collections import deque

# ─── Grid Legend ──────────────────────────────────────────────────────────────
# 0 = open cell   1 = wall   S = start   E = exit
# Items are uppercase letters A, B, C ... with assigned priorities/costs

DIRECTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

GRID = [
    ["S", 0,   0,   1,   0,   0,   0],
    [ 0,  1,   0,   1,   0,   1,   0],
    [ 0,  1,  "A",  0,   0,   1,  "B"],
    [ 0,  0,   0,   1,   0,   0,   0],
    [ 1,  1,   0,   0,  "C",  1,   0],
    [ 0,  0,   0,   1,   0,   0,   0],
    [ 0,  1,   0,   0,   0,   1,  "E"],
]

# Priority/reward of each item (higher = more valuable to collect first)
ITEM_PRIORITY = {"A": 3, "B": 5, "C": 2}

# Cost/weight to collect each item (used in UCS / A*)
ITEM_COST = {"A": 2, "B": 1, "C": 4}


# ─── Grid Utilities ───────────────────────────────────────────────────────────

def parse_grid(grid: list) -> dict:
    """Extract positions of start, exit, and all items from the grid."""
    info = {"start": None, "exit": None, "items": {}}
    for r, row in enumerate(grid):
        for c, cell in enumerate(row):
            if cell == "S":
                info["start"] = (r, c)
            elif cell == "E":
                info["exit"] = (r, c)
            elif isinstance(cell, str) and cell.isalpha():
                info["items"][cell] = (r, c)
    return info


def is_walkable(grid: list, r: int, c: int) -> bool:
    rows, cols = len(grid), len(grid[0])
    return 0 <= r < rows and 0 <= c < cols and grid[r][c] != 1


def neighbors(grid: list, pos: tuple) -> list:
    r, c = pos
    return [(r+dr, c+dc) for dr, dc in DIRECTIONS if is_walkable(grid, r+dr, c+dc)]


def manhattan(a: tuple, b: tuple) -> int:
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


# ─── Point-to-Point BFS (shortest path between two cells) ────────────────────

def bfs_path(grid: list, start: tuple, goal: tuple) -> list:
    """Return shortest path from start to goal using BFS."""
    queue = deque([(start, [start])])
    visited = {start}
    while queue:
        pos, path = queue.popleft()
        if pos == goal:
            return path
        for nb in neighbors(grid, pos):
            if nb not in visited:
                visited.add(nb)
                queue.append((nb, path + [nb]))
    return []


def bfs_cost(grid: list, start: tuple, goal: tuple) -> int:
    return max(0, len(bfs_path(grid, start, goal)) - 1)


# ─── 1. BFS — Collect all items in fixed order, then exit ────────────────────

def bfs_fixed_order(grid: list, info: dict) -> tuple:
    """
    BFS strategy: visit items in the order they appear (no prioritization),
    then reach exit. Returns (full_path, total_steps, items_order).
    """
    items_order = list(info["items"].keys())
    waypoints = [info["items"][i] for i in items_order] + [info["exit"]]
    pos = info["start"]
    full_path = [pos]
    total_steps = 0

    for wp in waypoints:
        segment = bfs_path(grid, pos, wp)
        if not segment:
            return None, float("inf"), items_order
        full_path += segment[1:]
        total_steps += len(segment) - 1
        pos = wp

    return full_path, total_steps, items_order


# ─── 2. DFS — Collect all items via DFS traversal order ──────────────────────

def dfs_order(grid: list, info: dict) -> tuple:
    """
    DFS strategy: explore items in DFS discovery order from start,
    then reach exit. Returns (full_path, total_steps, items_order).
    """
    items_positions = set(info["items"].values())
    visited = set()
    order_found = []

    stack = [info["start"]]
    while stack and len(order_found) < len(info["items"]):
        pos = stack.pop()
        if pos in visited:
            continue
        visited.add(pos)
        if pos in items_positions:
            for name, loc in info["items"].items():
                if loc == pos and name not in order_found:
                    order_found.append(name)
        for nb in neighbors(grid, pos):
            if nb not in visited:
                stack.append(nb)

    # Fill any items DFS missed
    for name in info["items"]:
        if name not in order_found:
            order_found.append(name)

    waypoints = [info["items"][i] for i in order_found] + [info["exit"]]
    pos = info["start"]
    full_path = [pos]
    total_steps = 0

    for wp in waypoints:
        segment = bfs_path(grid, pos, wp)
        if not segment:
            return None, float("inf"), order_found
        full_path += segment[1:]
        total_steps += len(segment) - 1
        pos = wp

    return full_path, total_steps, order_found




# ─── 3. Priority-Based Greedy — Collect highest priority items first ──────────

def greedy_priority(grid: list, info: dict) -> tuple:
    """
    Greedy strategy: always collect the highest-priority uncollected item next,
    then exit. Returns (full_path, total_steps, items_order).
    """
    remaining = dict(info["items"])
    pos = info["start"]
    full_path = [pos]
    total_steps = 0
    items_order = []

    while remaining:
        # Pick item with highest priority
        best = max(remaining, key=lambda k: ITEM_PRIORITY[k])
        target = remaining.pop(best)
        items_order.append(best)
        segment = bfs_path(grid, pos, target)
        if not segment:
            return None, float("inf"), items_order
        full_path += segment[1:]
        total_steps += len(segment) - 1
        pos = target

    segment = bfs_path(grid, pos, info["exit"])
    if not segment:
        return None, float("inf"), items_order
    full_path += segment[1:]
    total_steps += len(segment) - 1

    return full_path, total_steps, items_order


# ─── 4. UCS — Minimize total path steps + item collection costs ───────────────

def ucs(grid: list, info: dict) -> tuple:
    """
    Uniform Cost Search over item-collection orderings.
    Cost = path steps + sum of ITEM_COST for each collected item.
    State = (position, frozenset of collected items).
    Returns (full_path, total_cost, items_order).
    """
    all_items = frozenset(info["items"].keys())
    start_state = (info["start"], frozenset())
    # heap: (cost, position, collected, path_of_positions)
    heap = [(0, info["start"], frozenset(), [info["start"]])]
    visited = {}

    while heap:
        cost, pos, collected, path = heapq.heappop(heap)

        state = (pos, collected)
        if state in visited:
            continue
        visited[state] = cost

        # Reached exit with all items
        if pos == info["exit"] and collected == all_items:
            order = sorted(collected, key=lambda k: path.index(info["items"][k]))
            return path, cost, order

        for nb in neighbors(grid, pos):
            cell = grid[nb[0]][nb[1]]
            extra = 0
            new_collected = collected
            if isinstance(cell, str) and cell in info["items"] and cell not in collected:
                extra = ITEM_COST[cell]
                new_collected = collected | {cell}
            new_cost = cost + 1 + extra
            new_state = (nb, new_collected)
            if new_state not in visited:
                heapq.heappush(heap, (new_cost, nb, new_collected, path + [nb]))

    return None, float("inf"), []


# ─── 5. A* — Minimize cost with Manhattan heuristic to nearest goal ───────────

def astar_multi(grid: list, info: dict) -> tuple:
    """
    A* search over item-collection orderings.
    f(n) = g(n) + h(n)
    g(n) = path steps + item costs so far
    h(n) = Manhattan distance to nearest uncollected item or exit
    Returns (full_path, total_cost, items_order).
    """
    all_items = frozenset(info["items"].keys())

    def heuristic(pos, collected):
        remaining = [info["items"][k] for k in all_items if k not in collected]
        targets = remaining if remaining else [info["exit"]]
        return min(manhattan(pos, t) for t in targets)

    heap = [(0, 0, info["start"], frozenset(), [info["start"]])]
    visited = {}

    while heap:
        f, g, pos, collected, path = heapq.heappop(heap)

        state = (pos, collected)
        if state in visited:
            continue
        visited[state] = g

        if pos == info["exit"] and collected == all_items:
            order = sorted(collected, key=lambda k: path.index(info["items"][k]))
            return path, g, order

        for nb in neighbors(grid, pos):
            cell = grid[nb[0]][nb[1]]
            extra = 0
            new_collected = collected
            if isinstance(cell, str) and cell in info["items"] and cell not in collected:
                extra = ITEM_COST[cell]
                new_collected = collected | {cell}
            ng = g + 1 + extra
            new_state = (nb, new_collected)
            if new_state not in visited:
                h = heuristic(nb, new_collected)
                heapq.heappush(heap, (ng + h, ng, nb, new_collected, path + [nb]))

    return None, float("inf"), []


# ─── Render ───────────────────────────────────────────────────────────────────

def render(grid: list, path: list, label: str = "") -> None:
    path_set = set(path)
    start = path[0] if path else None
    end   = path[-1] if path else None
    rows  = len(grid)
    cols  = len(grid[0])

    print(f"  {label}")
    for r in range(rows):
        line = []
        for c in range(cols):
            pos  = (r, c)
            cell = grid[r][c]
            if pos == start:               line.append(" S ")
            elif pos == end:               line.append(" E ")
            elif isinstance(cell, str) and cell.isalpha() and cell not in ("S","E"):
                                           line.append(f"[{cell}]")
            elif pos in path_set:          line.append(" . ")
            elif cell == 1:                line.append("###")
            else:                          line.append("   ")
        print("  " + "".join(line))


# ─── Compare All Strategies ───────────────────────────────────────────────────

def compare(grid: list) -> None:
    info = parse_grid(grid)

    print("=" * 60)
    print("  GRID INFO")
    print(f"  Start : {info['start']}")
    print(f"  Exit  : {info['exit']}")
    for name, pos in info["items"].items():
        print(f"  Item {name}: pos={pos}  priority={ITEM_PRIORITY[name]}  cost={ITEM_COST[name]}")
    print("=" * 60)

    strategies = [
        ("BFS  (fixed order)",      bfs_fixed_order),
        ("DFS  (discovery order)",  dfs_order),
        ("Greedy (priority-based)", greedy_priority),
        ("UCS  (min step+cost)",    ucs),
        ("A*   (min step+cost+h)",  astar_multi),
    ]

    results = []
    for name, fn in strategies:
        path, cost, order = fn(grid, info)
        steps = len(path) - 1 if path else None
        results.append((name, path, cost, steps, order))

    for name, path, cost, steps, order in results:
        print(f"\n{name}")
        print("-" * 60)
        if path:
            render(grid, path, label="")
            print(f"  Item order  : {' -> '.join(order)}")
            print(f"  Path steps  : {steps}")
            print(f"  Total cost  : {cost}")
        else:
            print("  No solution found.")

    print("\nCOMPARISON TABLE")
    print("-" * 60)
    print(f"  {'Strategy':<28} {'Steps':>6} {'Cost':>6} {'Order':>12}")
    print(f"  {'-' * 56}")
    for name, path, cost, steps, order in results:
        s = str(steps) if steps is not None else "N/A"
        c = f"{cost:.0f}" if cost != float("inf") else "N/A"
        o = "->".join(order) if order else "N/A"
        print(f"  {name:<28} {s:>6} {c:>6} {o:>12}")

    print("\nTRADE-OFF ANALYSIS")
    print("-" * 60)
    valid = [(n, p, c, s, o) for n, p, c, s, o in results if s is not None]
    if valid:
        min_steps = min(valid, key=lambda x: x[3])
        min_cost  = min(valid, key=lambda x: x[2])
        print(f"  Shortest path : {min_steps[0]} ({min_steps[3]} steps)")
        print(f"  Lowest cost   : {min_cost[0]}  (cost={min_cost[2]:.0f})")
        print()
        print("  BFS/DFS  -> ignores costs, minimizes hops only")
        print("  Greedy   -> fast but suboptimal (no global view)")
        print("  UCS      -> optimal cost, explores more nodes")
        print("  A*       -> optimal cost, fewer nodes via heuristic")
    print("=" * 60)


if __name__ == "__main__":
    compare(GRID)

  GRID INFO
  Start : (0, 0)
  Exit  : (6, 6)
  Item A: pos=(2, 2)  priority=3  cost=2
  Item B: pos=(2, 6)  priority=5  cost=1
  Item C: pos=(4, 4)  priority=2  cost=4

BFS  (fixed order)
------------------------------------------------------------
  
   S  .  . ###         
     ### . ###   ###   
     ###[A] .  . ###[B]
           ### .  .  . 
  ######      [C]###   
           ### .  .  . 
     ###         ### E 
  Item order  : A -> B -> C
  Path steps  : 18
  Total cost  : 18

DFS  (discovery order)
------------------------------------------------------------
  
   S  .  . ###         
     ### . ###   ###   
     ###[A]      ###[B]
         . ### .  .  . 
  ###### .  . [C]### . 
           ###       . 
     ###         ### E 
  Item order  : A -> C -> B
  Path steps  : 16
  Total cost  : 16

Greedy (priority-based)
------------------------------------------------------------
  
   S  .  . ###         
     ### . ###   ###   
     ###[A] .  . ###[B]
         . ### .  .  . 
  ####